SECTION 1 — Install Dependencies

In [ ]:
!apt-get install -y aria2
!pip install requests tqdm msal cryptography pymysql

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.5/121.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 2.7 MB/s eta 0:00:00


SECTION 2 — Imports

In [ ]:
import os
import time
import pymysql
import requests
import subprocess
import urllib.parse

from tqdm import tqdm
from cryptography.fernet import Fernet
from msal import PublicClientApplication
from google.colab import userdata
from concurrent.futures import ThreadPoolExecutor, as_completed

SECTION 3 — Load Colab Secrets

In [ ]:
CLIENT_ID = userdata.get('CLIENT_ID')

DB_HOST = userdata.get('DB_HOST')
DB_USER = userdata.get('DB_USER')
DB_PASSWORD = userdata.get('DB_PASSWORD')
DB_NAME = userdata.get('DB_NAME')

FERNET_KEY = userdata.get('FERNET_KEY')

SECTION 4 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


SECTION 5 — Secure MySQL SSL Connection

In [ ]:
db = pymysql.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME,

    ssl={
        'ca': '/content/drive/MyDrive/certs/cacert-2023-08-22.pem'
    }
)

cursor = db.cursor()

print("Secure SSL MySQL connection established")

Secure SSL MySQL connection established


SECTION 6 — Encryption Setup

In [ ]:
fernet = Fernet(
    FERNET_KEY.encode()
)

SECTION 7 — Database Functions

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS onedrive_tokens (
    id INT PRIMARY KEY AUTO_INCREMENT,
    encrypted_refresh_token TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

db.commit()


def save_refresh_token(refresh_token):

    encrypted = fernet.encrypt(
        refresh_token.encode()
    ).decode()

    cursor.execute("DELETE FROM onedrive_tokens")

    cursor.execute(
        """
        INSERT INTO onedrive_tokens
        (encrypted_refresh_token)
        VALUES (%s)
        """,
        (encrypted,)
    )

    db.commit()


def get_refresh_token():

    cursor.execute("""
        SELECT encrypted_refresh_token
        FROM onedrive_tokens
        LIMIT 1
    """)

    row = cursor.fetchone()

    if not row:
        return None

    encrypted = row[0]

    decrypted = fernet.decrypt(
        encrypted.encode()
    ).decode()

    return decrypted

SECTION 8 — Authentication System

In [ ]:
SCOPES = [
    "Files.ReadWrite.All"
]


def device_login():

    app = PublicClientApplication(
        CLIENT_ID,
        authority="https://login.microsoftonline.com/common"
    )

    flow = app.initiate_device_flow(
        scopes=SCOPES
    )

    print(flow["message"])

    result = app.acquire_token_by_device_flow(flow)

    if "access_token" not in result:
        raise Exception("Authentication failed")

    if "refresh_token" in result:
        save_refresh_token(
            result["refresh_token"]
        )

    return result["access_token"]


def authenticate():

    refresh_token = get_refresh_token()

    if refresh_token:

        print("Using stored refresh token")

        token_url = (
            "https://login.microsoftonline.com/"
            "common/oauth2/v2.0/token"
        )

        data = {
            "client_id": CLIENT_ID,
            "grant_type": "refresh_token",
            "refresh_token": refresh_token,
            "scope": " ".join(SCOPES)
        }

        response = requests.post(
            token_url,
            data=data
        )

        result = response.json()

        if "access_token" in result:

            print("Refresh token valid")

            if "refresh_token" in result:
                save_refresh_token(
                    result["refresh_token"]
                )

            return result["access_token"]

        print("Refresh token invalid")

    return device_login()

SECTION 9 — Link Downloader

In [ ]:
def download_file(url):
    # Parse the filename cleanly from the URL
    filename = urllib.parse.unquote(url.split("/")[-1].split("?")[0])

    print(f"Initiating 16-thread download for: {filename}...")
    start_time = time.time()

    # The aria2c command arguments
    command = [
        "aria2c",
        "--console-log-level=error", # Hides unnecessary debug text
        "--summary-interval=1",      # Shows progress every 1 second
        "-x", "16",                  # Maximum number of connections to the server
        "-s", "16",                  # Split the file into 16 concurrent chunks
        "-k", "1M",                  # Minimum chunk size
        "-o", filename,              # Save as this filename
        url
    ]

    try:
        # Execute the command in Colab's terminal
        subprocess.run(command, check=True)
    except subprocess.CalledProcessError as e:
        print(f"Download failed: {e}")
        raise

    elapsed = time.time() - start_time
    print(f"\nDownload completed in {elapsed:.2f} seconds!")

    return filename

SECTION 10 — OneDrive Folder Management

In [ ]:
UPLOAD_FOLDER_NAME = "OneUploader"


def get_or_create_folder(access_token):

    headers = {
        "Authorization":
            f"Bearer {access_token}"
    }

    response = requests.get(
        "https://graph.microsoft.com/v1.0/me/drive/root/children",
        headers=headers
    )

    items = response.json().get("value", [])

    for item in items:

        if item["name"] == UPLOAD_FOLDER_NAME:
            return item["id"]

    payload = {
        "name": UPLOAD_FOLDER_NAME,
        "folder": {}
    }

    response = requests.post(
        "https://graph.microsoft.com/v1.0/me/drive/root/children",
        headers=headers,
        json=payload
    )

    return response.json()["id"]

SECTION 11 — Upload System

In [ ]:
CHUNK_SIZE = 10 * 1024 * 1024

def upload_file(access_token, folder_id, file_path):
    file_size = os.path.getsize(file_path)
    filename = os.path.basename(file_path)

    # 1. Create the Upload Session
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    endpoint = (
        f"https://graph.microsoft.com/v1.0/me/drive/items/"
        f"{folder_id}:/{filename}:/createUploadSession"
    )

    response = requests.post(endpoint, headers=headers, json={})
    response.raise_for_status()
    upload_url = response.json()["uploadUrl"]

    # 2. Calculate the chunks
    chunks = []
    for i in range(0, file_size, CHUNK_SIZE):
        chunk_start = i
        chunk_end = min(i + CHUNK_SIZE - 1, file_size - 1)
        chunk_length = (chunk_end - chunk_start) + 1
        chunks.append((chunk_start, chunk_end, chunk_length))

    progress = tqdm(
        total=file_size,
        unit='B',
        unit_scale=True,
        desc=f"Uploading {filename}"
    )

    start_time = time.time()
    uploaded_bytes = 0

    # 3. Define the worker function for a single chunk
    def upload_chunk(start, end, length):
        with open(file_path, "rb") as f:
            f.seek(start)
            data = f.read(length)

        chunk_headers = {
            "Content-Length": str(length),
            "Content-Range": f"bytes {start}-{end}/{file_size}"
        }

        # Use a generous timeout for data center connections
        r = requests.put(upload_url, headers=chunk_headers, data=data, timeout=120)

        if r.status_code not in [200, 201, 202]:
            raise Exception(f"Chunk upload failed with status {r.status_code}: {r.text}")

        return length

    # 4. Execute the uploads in parallel
    # Max workers set to 4 to respect Microsoft Graph API throttling limits
    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = {
            executor.submit(upload_chunk, start, end, length): length
            for start, end, length in chunks
        }

        for future in as_completed(futures):
            try:
                length_completed = future.result()
                uploaded_bytes += length_completed
                progress.update(length_completed)

                elapsed = time.time() - start_time
                speed = uploaded_bytes / elapsed if elapsed > 0 else 0

                progress.set_postfix({
                    "Speed": f"{speed/1024/1024:.2f} MB/s"
                })
            except Exception as e:
                print(f"\nError during upload: {e}")
                # Note: In a production script, you'd want logic here to cancel
                # remaining futures if a fatal error occurs.

    progress.close()
    print("Upload completed")

SECTION 12 — Run

In [ ]:
db = pymysql.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME,

    ssl={
        'ca': '/content/drive/MyDrive/certs/cacert-2023-08-22.pem'
    }
)

cursor = db.cursor()

url = input("Direct Download URL: ")

access_token = authenticate()

folder_id = get_or_create_folder(
    access_token
)

dowloaded_file = download_file(url)

upload_file(
    access_token,
    folder_id,
    dowloaded_file
)

To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code EVME67T97 to authenticate.


Uploading Mortal.Kombat.II.2026.1080p.DCPRIP.H264-DKS2.mkv.zip: 100%|██████████| 6.88G/6.88G [18:29<00:00, 6.20MB/s, Speed=5.91 MB/s, ETA=0.0s]

Upload completed
